# Lab 6.7 &mdash; Memory: Conversation History

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 3 &middot; Module 6 &mdash; Frameworks for Building AI Agents**

### What you'll do
- Store each turn (role, text) in a memory object
- Format the running history the model re-reads each turn
- Ask a real elliptical follow-up and watch memory resolve it

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. Read the **Concept**, fill the real `___` blanks in **Build it** (real tool bodies, real prompts, the real `create_agent` call), then **Run it for real** &mdash; a real LLM drives a real agent over real tools &mdash; and **read the trace** to see exactly what it did. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain (`langchain`, `langchain-core`, `langchain-ollama`, `langgraph`) and a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`). If Ollama is down, the run cells print how to start it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* aborts the whole agent run (you'll see exactly this in Lab 11).

**Reference:** [Module 6 slides &mdash; Memory -- plugged in, not hand-built](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 6 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # SERPER_API_KEY, WOLFRAM_ALPHA_APPID, GROQ/OPENAI keys

WORK = "/tmp/biaa-lab-06-07"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model. Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
A framework keeps **conversation memory** so follow-ups work: ask *"capital of France?"*, then
*"is it bigger than Osaka?"* &mdash; the model needs the earlier turns to resolve *"it"*. Memory is a
component you **attach**: it stores turns and re-injects the **history** into each prompt. Here you build
it and prove it with a **real** two-turn conversation. (Long-term memory is a vector store &mdash; the
bridge to RAG.)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# LangChain represents a turn as a message; the running list is the history:
history = [HumanMessage(content="What is the capital of France?"), AIMessage(content="Paris.")]
print("a turn:", (history[0].type, history[0].content))

## Build it
Implement `ConversationMemory` (store + format the history) and a prompt that injects it.

In [ ]:
from langchain_core.prompts import PromptTemplate

class ConversationMemory:
    def __init__(self):
        self.turns = []
    def add(self, role, text):
        self.turns.append(___)   # TODO: store the turn as (role, text)
    def history(self):
        return ___   # TODO: one "role: text" line per turn, newline-joined

def build_prompt(memory, question):
    template = PromptTemplate.from_template("Conversation so far:\n{history}\nUser: {input}\nAnswer concisely.")
    return template.format(history=___, input=question)   # TODO: the running history

try:
    mem = ConversationMemory()
    mem.add("user", "What is the capital of France?")
    mem.add("assistant", "Paris.")
    print(build_prompt(mem, "Is it bigger than Osaka?"))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run a real two-turn conversation. The second question says *'it'* &mdash; only memory lets the model resolve it to Paris.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        mem = ConversationMemory()
        q1 = "What is the capital of France?"
        a1 = llm.invoke(build_prompt(mem, q1)).content
        print("Q1:", q1); print("A1:", a1)
        mem.add("user", q1); mem.add("assistant", a1)

        q2 = "Is it bigger than Osaka?"     # 'it' only resolves via memory
        a2 = llm.invoke(build_prompt(mem, q2)).content
        print("Q2:", q2); print("A2:", a2)
        print("---")
        print("memory carried the context?", "paris" in a2.lower() or "france" in a2.lower())
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- With memory, the model answers Q2 **about Paris** even though Q2 never says "Paris" &mdash; it read the history.
- Try deleting the `mem.add(...)` line after A1: the model no longer knows what *"it"* is. That's the whole value of memory.
- In a framework this is a component you attach, not a scratchpad you re-thread by hand.

## Your turn (open task &mdash; no grader)
Extend the conversation to a third turn (e.g. *"And what language do they speak there?"*) &mdash; add each
answer to `mem` and re-run. **What good looks like:** each new answer stays on-topic across turns; remove
memory and the follow-ups fall apart.

---
### Key takeaway
Memory carries context across turns so 'is it bigger than Osaka?' just works. In a framework it's a component you attach -- and you just proved it with a real model.

[&#8592; All Module 6 labs](./index.html) &nbsp;&middot;&nbsp; [Module 6 slides](../../presentation/day3-module6-frameworks-for-building-ai-agents.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>